# NBA 3.1 HCP Universe — Construction

This notebook builds the full HCP Universe by constructing and combining 4 criteria:

* **C1:** Core Prescribers — indication × drug pool deciling on `lrx_nbrx`
* **C2:** Kerendia Writers — all HCPs with NBRX > 0 (all claim types, full history)
* **C3:** Specialists — Nephrologists + Cardiologists
* **C4:** Targeting/BPT — HCPs on commercial targeting list (Snowflake)

**Target:** ≤ 300,000 HCPs | Maximize Kerendia writer capture (61,583 writers)

In [0]:
%run "/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
# ============================================================
# Imports & Parameters
# ============================================================
import builtins
from functools import reduce
from pyspark.sql import functions as F, Window

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA  = "pace_ker_in_house_schema"
SOURCE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx"

LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"

INDICATIONS = ["T2D", "CKD", "HF", "T1D"]
DRUG_POOLS = {
    "SGLT2_GLP1": ["SGLT2", "GLP1"],
    "ACE_ARB":    ["ACE", "ARB"],
}

INDICATION_COL = "INDICATION_GROUP_APP1"
PRODUCT_COL    = "PRODUCT_CLASS"
METRIC_COL     = "NBRX"

KERENDIA_NBRX_PATH = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nbrx_from_raw_fct_rx"
)

NBA30_PATH = (
    f"/Volumes/{CATALOG}/"
    "kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
    "dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)

MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"

print("Parameters loaded.")

In [0]:
# ============================================================
# Load lrx_nbrx with lookback filter
# ============================================================
df_src = spark.table(SOURCE_TABLE)
lrx = df_src.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)
print(f"Lookback rows:  {lrx.count():,}")
print(f"Unique HCPs:    {lrx.select('PROVIDER_ID').distinct().count():,}")

In [0]:
# ============================================================
# Cumulative volume-based decile function (NBRX)
# Each decile ≈ 10% of total NBRX volume.
# D1 = lowest-volume HCPs, D10 = highest-volume.
# Tiebreaker: PROVIDER_ID ascending.
# ============================================================
def compute_deciles(df, indication, pool_products):
    """
    For a given indication × drug pool, compute cumulative
    volume-based deciles (1–10) per PROVIDER_ID.
    Returns (DataFrame[PROVIDER_ID, TOTAL_NBRX, DECILE], total_hcps, total_vol).
    """
    filtered = df.filter(
        (F.col(INDICATION_COL).contains(indication)) &
        (F.col(PRODUCT_COL).isin(pool_products))
    )
    hcp_vol = filtered.groupBy("PROVIDER_ID").agg(
        F.sum(METRIC_COL).alias("TOTAL_NBRX")
    )
    total_vol = hcp_vol.agg(F.sum("TOTAL_NBRX")).collect()[0][0]
    if total_vol is None or total_vol == 0:
        return None, 0, 0
    vol_per_decile = float(total_vol) / 10.0

    w = Window.orderBy(F.asc("TOTAL_NBRX"), F.asc("PROVIDER_ID"))
    hcp_vol = hcp_vol.withColumn("CUM_NBRX", F.sum("TOTAL_NBRX").over(w))
    hcp_vol = hcp_vol.withColumn(
        "DECILE",
        F.least(
            (F.col("CUM_NBRX") / F.lit(vol_per_decile)).cast("int") + 1,
            F.lit(10)
        )
    ).drop("CUM_NBRX")

    total_hcps = hcp_vol.count()
    return hcp_vol, total_hcps, int(total_vol)

print("Decile function defined.")

In [0]:
# ============================================================
# Compute all 8 decile sets with full D1–D10 distribution
# ============================================================
all_decile_dfs = {}   # (indication, pool_name) -> decile DataFrame
all_stats = []        # summary dicts

for ind in INDICATIONS:
    for pool_name, pool_products in DRUG_POOLS.items():
        decile_df, n_hcps, total_vol = compute_deciles(lrx, ind, pool_products)
        if decile_df is None:
            print(f"{ind} × {pool_name}: No data")
            continue
        
        all_decile_dfs[(ind, pool_name)] = decile_df
        
        # Print decile distribution
        print(f"\n{'='*75}")
        print(f"{ind} × {pool_name}  |  Total HCPs: {n_hcps:,}  |  Total NBRx: {total_vol:,}")
        print(f"{'='*75}")
        print(f"{'Decile':>8} {'HCPs':>10} {'%HCPs':>8} {'NBRx':>12} {'Cum%HCPs':>10} {'Cum%Vol':>10}")
        print("-" * 60)
        
        decile_stats = (
            decile_df.groupBy("DECILE")
            .agg(
                F.count("*").alias("N_HCPS"),
                F.sum("TOTAL_NBRX").alias("DEC_NBRX")
            )
            .orderBy("DECILE")
            .collect()
        )
        
        cum_hcps = 0
        cum_vol = 0
        for row in decile_stats:
            cum_hcps += row["N_HCPS"]
            cum_vol += row["DEC_NBRX"]
            print(f"D{row['DECILE']:>6}  {row['N_HCPS']:>10,}  {100*row['N_HCPS']/n_hcps:>7.1f}%"
                  f"  {row['DEC_NBRX']:>11,.0f}  {100*cum_hcps/n_hcps:>9.1f}%  {100*cum_vol/total_vol:>9.1f}%")
        
        # Store D2+ count
        d2plus = decile_df.filter(F.col("DECILE") >= 2).count()
        all_stats.append({
            "indication": ind, "pool": pool_name,
            "total_hcps": n_hcps, "total_vol": total_vol,
            "d2_plus": d2plus
        })

print(f"\n\nAll 8 sets computed. {len(all_decile_dfs)} DataFrames cached.")

In [0]:
# ============================================================
# Summary table: HCP counts by decile cutoff (D2+ through D5+)
# ============================================================
print(f"{'Ind × Pool':<25} {'Total':>10} {'D2+':>10} {'D3+':>10} {'D4+':>10} {'D5+':>10}")
print("=" * 70)
for (ind, pool_name), df in all_decile_dfs.items():
    total = df.count()
    cuts = []
    for c in [2, 3, 4, 5]:
        cuts.append(df.filter(F.col("DECILE") >= c).count())
    print(f"{ind} × {pool_name:<18} {total:>10,} {cuts[0]:>10,} {cuts[1]:>10,} {cuts[2]:>10,} {cuts[3]:>10,}")

In [0]:
# ============================================================
# Core C1 = Set 1 (T2D×SGLT2/GLP1) ∪ Set 3 (CKD×SGLT2/GLP1) at D2+
# ============================================================
set1 = all_decile_dfs[("T2D", "SGLT2_GLP1")]
set3 = all_decile_dfs[("CKD", "SGLT2_GLP1")]

c1_core = (
    set1.filter(F.col("DECILE") >= 2).select("PROVIDER_ID")
    .unionByName(
        set3.filter(F.col("DECILE") >= 2).select("PROVIDER_ID")
    )
    .distinct()
)
c1_ct = c1_core.count()
print(f"C1 Core (Set1 ∪ Set3, D2+): {c1_ct:,} HCPs")

In [0]:
# ============================================================
# C2: Kerendia Writers (NBRX > 0, all claim types, full history)
# ============================================================
ker_fct_rx = spark.read.format("delta").load(KERENDIA_NBRX_PATH)
ker_fct_rx = ker_fct_rx.withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))

c2_writers = ker_fct_rx.filter(F.col("NBRX_KERENDIA_OVERALL") > 0).select("PROVIDER_ID").distinct()
c2_ct = c2_writers.count()
print(f"C2 Kerendia Writers: {c2_ct:,}")

In [0]:
# ============================================================
# C3: Nephrologists + Cardiologists from lrx_nbrx
# ============================================================
specialty_hcps = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct()

neph_c3 = specialty_hcps.filter(F.col("SPECIALTY_GROUP") == "NEPH").select("PROVIDER_ID").distinct()
card_c3 = specialty_hcps.filter(F.col("SPECIALTY_GROUP") == "CARD").select("PROVIDER_ID").distinct()
c3_specialists = neph_c3.unionByName(card_c3).distinct()

neph_ct = neph_c3.count()
card_ct = card_c3.count()
c3_ct = c3_specialists.count()

print(f"C3 NEPH: {neph_ct:,}")
print(f"C3 CARD: {card_ct:,}")
print(f"C3 Total (NEPH ∪ CARD): {c3_ct:,}")

In [0]:
# ============================================================
# C4: Targeting/BPT HCPs from Snowflake CP_KER_HCP
# Map BH_ID -> PROVIDER_ID via mapping table
# ============================================================
query_c4 = """
    SELECT DISTINCT BAYER_ID as BH_ID
    FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
    WHERE FF IN ('CR1','CR2','CR4','CR5')
    AND ((UPPER(PRODUCT_1)='KERENDIA') OR (UPPER(PRODUCT_2)='KERENDIA'))
"""

try:
    c4_raw = Get_Data_Snowflakes(query_c4)
    c4_bhids = c4_raw.select(F.col("BH_ID").cast("string").alias("BH_ID")).distinct()
    c4_bhid_ct = c4_bhids.count()
    print(f"C4 BH_IDs from Snowflake: {c4_bhid_ct:,}")

    # Map BH_ID -> PROVIDER_ID
    id_map = spark.table(MAPPING_TABLE).select(
        F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"),
        F.col("BH_ID").cast("string").alias("BH_ID")
    )
    c4_mapped = c4_bhids.join(id_map, on="BH_ID", how="inner").select("PROVIDER_ID").distinct()
    c4_ct = c4_mapped.count()
    print(f"C4 mapped PROVIDER_IDs: {c4_ct:,}")
except Exception as e:
    print(f"WARNING: Snowflake query failed: {e}")
    print("Proceeding without C4. Universe will be C1+C2+C3 only.")
    c4_mapped = spark.createDataFrame([], "PROVIDER_ID: string")
    c4_ct = 0

In [0]:
# ============================================================
# Combine all criteria into the final universe
# ============================================================
universe = (
    c1_core
    .unionByName(c2_writers)
    .unionByName(c3_specialists)
    .unionByName(c4_mapped)
    .distinct()
)
universe_ct = universe.count()

print(f"\nUniverse Construction:")
print(f"  C1 (Core D2+):       {c1_ct:,}")
print(f"  C2 (Kerendia):       {c2_ct:,}")
print(f"  C3 (NEPH+CARD):      {c3_ct:,}")
print(f"  C4 (Targeting):      {c4_ct:,}")
print(f"  ----------------------------")
print(f"  Total Universe:      {universe_ct:,}")
print(f"  Budget check:        {'PASS ✅' if universe_ct <= 300000 else 'FAIL ❌'} (≤ 300K)")

In [0]:
# ============================================================
# Show incremental buildup: how each criterion adds HCPs
# ============================================================
print(f"{'Step':<30} {'Total':>10} {'Incremental':>12}")
print("-" * 55)

running = c1_core
running_ct = c1_ct
print(f"{'C1 Core (Set1∪Set3, D2+)':<30} {running_ct:>10,} {'-':>12}")

for label, criterion_df in [("+ C2 Kerendia Writers", c2_writers),
                             ("+ C3 NEPH", neph_c3),
                             ("+ C3 CARD", card_c3),
                             ("+ C4 Targeting/BPT", c4_mapped)]:
    prev = running_ct
    running = running.unionByName(criterion_df).distinct()
    running_ct = running.count()
    incr = running_ct - prev
    print(f"{label:<30} {running_ct:>10,} {'+' + f'{incr:,}':>12}")

In [0]:
# ============================================================
# Create flagged universe DataFrame
# ============================================================
universe_flagged = (
    universe
    .join(c1_core.withColumn("C1_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .join(c2_writers.withColumn("C2_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .join(c3_specialists.withColumn("C3_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .join(c4_mapped.withColumn("C4_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .fillna(0, subset=["C1_FLAG", "C2_FLAG", "C3_FLAG", "C4_FLAG"])
)

# Add specialty from lrx_nbrx
spec_map = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct()
# Take first non-null specialty per provider
spec_dedup = spec_map.groupBy("PROVIDER_ID").agg(
    F.first("SPECIALTY_GROUP", ignorenulls=True).alias("SPECIALTY_GROUP")
)
universe_flagged = universe_flagged.join(spec_dedup, on="PROVIDER_ID", how="left")

print(f"Flagged universe: {universe_flagged.count():,} HCPs")
print(f"\nFlag distribution:")
for flag in ["C1_FLAG", "C2_FLAG", "C3_FLAG", "C4_FLAG"]:
    ct = universe_flagged.filter(F.col(flag) == 1).count()
    print(f"  {flag} = 1: {ct:,} ({100*ct/universe_ct:.1f}%)")

print(f"\nSample:")
universe_flagged.show(10, truncate=False)

In [0]:
# ============================================================
# Verify: all C2 Kerendia writers are in the universe
# ============================================================
ker_in_universe = c2_writers.join(universe, on="PROVIDER_ID", how="inner").count()
ker_not_in = c2_writers.join(universe, on="PROVIDER_ID", how="left_anti").count()

print(f"Kerendia writers in universe:  {ker_in_universe:,} / {c2_ct:,} ({100*ker_in_universe/c2_ct:.1f}%)")
print(f"Kerendia writers NOT captured: {ker_not_in:,}")
assert ker_not_in == 0, f"ERROR: {ker_not_in} Kerendia writers missing from universe!"
print("\n✅ All Kerendia writers captured (100%).")

## Universe Construction Summary

The universe is built by unioning:

* **C1:** Core prescribers (Set1 T2D×SGLT2/GLP1 ∪ Set3 CKD×SGLT2/GLP1 at D2+)
* **C2:** All Kerendia writers (61,583 — NBRX > 0, all claim types, full history)
* **C3:** Nephrologists + Cardiologists
* **C4:** Targeting/BPT HCPs (from Snowflake CP_KER_HCP)

**Key outputs:**
* `universe_flagged` — DataFrame with PROVIDER_ID, C1–C4 flags, SPECIALTY_GROUP
* All Kerendia writers captured at 100%

**Next:** `03_Validation_Hotspot` — validate against NBA 3.0 baseline, compute M1–M4 metrics, profile missed writers.